## 1. INSTALLS, IMPORTS, AND PARAMETERS

### --- 1. Installs ---

In [5]:
!pip install geopandas geojson overpass alphashape torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.6/507.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 736.5/736.5 kB 46.2 MB/s eta 0:00:00


### --- 2. Imports ---

In [6]:
import pandas as pd
import json
import geojson
import overpass
import csv
import os
import logging
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from shapely.geometry import Point, Polygon
import requests
import time
from google.colab import drive
import sys
import ast
from alphashape import alphashape
import geopandas as gpd
import os
from collections import defaultdict
import os.path
from tqdm import tqdm
import numpy as np
import pickle
import random
import json
from itertools import permutations
import time as time_module
import networkx as nx
from datetime import datetime
import geopy.distance
import heapq
import math
import heapq
import pickle
import os

# (New imports for PyTorch / RL)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import copy
from collections import namedtuple, deque

### --- 3. Global Parameters ---

In [7]:
state_name = "Ohio"
city_name = "Columbus"
mini = True

### 1. Setup

In [8]:
# --- 4. Connect to Drive ---
try:
    drive.mount('/content/gdrive/', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

Mounted at /content/gdrive/
Google Drive mounted successfully.


In [9]:
# --- 5. Set Directories ---
try:
    personal_dir = "./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/"
    data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

    # We will save our new MADDPG models and results in a new folder
    output_dir = data_dir + "/UMST Graph/maddpg_baseline"

    os.makedirs(output_dir, exist_ok=True)
    print(f"Data directory: {data_dir}")
    print(f"Output directory: {output_dir}")

    # See directory content
    data = os.listdir(data_dir)
    print(f"Files in data_dir: {data}")

except Exception as e:
    print(f"Error setting up directories. Is your drive mounted and the path correct?")
    print(f"Personal Dir Path: {personal_dir}")
    print(f"Data Dir Path: {data_dir}")

Data directory: ./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data
Output directory: ./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data/UMST Graph/maddpg_baseline
Files in data_dir: ['avg_hotspot_data.json', 'results_rangewise', 'UMST Graph', 'results_randomized', 'results_fixed_buffer', 'results_multiply', 'Processed Location Data', 'Q Tables', 'Images', 'simulation_results', 'Original Location Data', 'gh_cache', 'Census Data', 'Hotspot Data', 'Deliveries']


# 2. RE-USABLE DATA & ENVIRONMENT CLASSES (THE "WORLD")
 This is to keep the deliveries same as they were even when operating on two different methods. This follows the same structure as used in the file `V7_02_order_bundling.ipynb`

#### 1. Delivery Class


In [13]:
INTERVALS = [900, 1200, 1500, 1800, 2700, 3600, 5400, 7200, 10800, 14400]

In [14]:
class Delivery:
    """
    Represents a single delivery request.
    (This is the simplified version from your notebook)
    """
    id_counter = 0

    def __init__(self, start_node, end_node, start_time, time_tolerance_factor,
                 shortest_path=None):
        # Identity
        self.id = Delivery.id_counter
        Delivery.id_counter += 1

        # Route (immutable)
        self.start_node = start_node
        self.end_node = end_node
        self.shortest_path = shortest_path if shortest_path else [start_node, end_node]

        # Time constraints
        self.start_time = start_time
        self.TIME_TOLERANCE_FACTOR = time_tolerance_factor
        self.time_limit = None  # Set after initialization

        # Current state (mutable)
        self.current_node = start_node
        self.path_index = 0
        self.in_transition = False
        self.time_till_next_node = 0
        self.wait_time_remaining = 0 # Used for heuristic baseline, but good to keep

        # Completion tracking
        self.completed = False
        self.successful = False
        self.end_time = None

        # Statistics
        self.distance_traveled = 0
        self.actual_path = [start_node]
        self.num_vehicle_changes = 0 # We'll re-purpose this for the RL agent
        self.times_bundled = 0

        # Bundle reference (managed by Bundle class or RL Env)
        self.current_bundle_id = None

    def get_next_node(self):
        """Get next node in planned path."""
        if self.path_index < len(self.shortest_path) - 1:
            return self.shortest_path[self.path_index + 1]
        return None

    def move_to_next_node(self, distance, travel_time):
        """Start transition to next node."""
        self.in_transition = True
        self.time_till_next_node = travel_time
        self.distance_traveled += distance

    def arrive_at_node(self, node):
        """Complete arrival at node."""
        self.in_transition = False
        self.current_node = node
        self.actual_path.append(node)
        self.path_index += 1

        # CRITICAL: Check if reached destination
        if self.current_node == self.end_node:
            self.completed = True
            # Note: successful status set later

    def reset(self):
        """Reset to initial state."""
        self.current_node = self.start_node
        self.path_index = 0
        self.in_transition = False
        self.time_till_next_node = 0
        self.wait_time_remaining = 0
        self.completed = False
        self.successful = False
        self.end_time = None
        self.distance_traveled = 0
        self.actual_path = [self.start_node]
        self.num_vehicle_changes = 0
        self.current_bundle_id = None

    def __repr__(self):
        status = "✅" if self.completed else ("🚗" if self.in_transition else "⏳")
        bundle = f"[B{self.current_bundle_id}]" if self.current_bundle_id else ""
        return f"D{self.id}:{self.start_node}→{self.end_node}|{status}{bundle}@N{self.current_node}"

#### 2. Graph & Pathfinding Helpers

In [15]:
def build_adjacency_matrix(graph: nx.Graph, tract_to_index, dist_attr='distance', time_attr='time'):
    """
    Create adjacency dict for direct edges only using integer indices.
    Each node maps to a list of (neighbor_index, distance, time_seconds)
    """
    adjacency = {}

    for u, v, data in graph.edges(data=True):
        u_idx = tract_to_index[u]
        v_idx = tract_to_index[v]
        dist = float(data.get(dist_attr, 0))
        time = float(data.get(time_attr, 0)) * 60  # minutes -> seconds
        adjacency.setdefault(u_idx, []).append((v_idx, dist, time))
        adjacency.setdefault(v_idx, []).append((u_idx, dist, time))

    num_edges = sum(len(v) for v in adjacency.values()) // 2
    print(f"✓ Adjacency built: {len(adjacency)} nodes with {num_edges} edges")
    return adjacency


def astar_shortest_path(adjacency, start, goal, node_positions=None, weight_type='time'):
    """
    Compute shortest path from start → goal using A*.
    - adjacency: dict[index] -> list of (neighbor_index, dist, time)
    - node_positions: dict[index] -> (lat, lon) for heuristic
    - weight_type: 'time' or 'distance'
    Returns: (total_cost, path_indices)
    """
    def heuristic(u, v):
        if node_positions is None:
            return 0
        (x1, y1), (x2, y2) = node_positions[u], node_positions[v]
        return math.hypot(x1 - x2, y1 - y2)

    frontier = [(0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0}

    while frontier:
        _, current = heapq.heappop(frontier)
        if current == goal:
            break

        if adjacency.get(current) is None:
            continue # Node has no outgoing edges

        for neighbor, dist, time in adjacency.get(current, []):
            weight = time if weight_type == 'time' else dist
            new_cost = cost_so_far[current] + weight

            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                priority = new_cost + heuristic(neighbor, goal)
                heapq.heappush(frontier, (priority, neighbor))
                came_from[neighbor] = current

    # Reconstruct path
    if goal not in came_from:
        return float('inf'), []

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()

    return cost_so_far[goal], path

#### 3. DeliveryList Class (The Task Generator)

(This is UNCHANGED from the original file)
This is CRUCIAL for ensuring we test both models on the *exact same* deliveries.

In [16]:
class DeliveryList:
    """Manages collection of deliveries with shortest path calculation."""

    def __init__(self, max_delivery_time, load, time_tolerance_factor,
                 hours=1, peaks=[0.25, 0.75], sigma=10, num_hotspots=50,
                 adjacency_matrix=None, node_positions=None):

        self.load = load
        self.hours = hours
        self.peaks = peaks
        self.sigma = sigma
        self.time_tolerance_factor = time_tolerance_factor
        self.max_delivery_time = max_delivery_time
        self.num_hotspots = num_hotspots
        self.adjacency_matrix = adjacency_matrix
        self.node_positions = node_positions

        if adjacency_matrix is None or node_positions is None:
            raise ValueError("adjacency_matrix and node_positions are required!")

        print("📊 Generating temporal distribution...")
        self.distribution, self.loads = self.generate_distribution()

        print("🗺️  Pre-computing all shortest paths...")
        self.precomputed_paths = {}  # (start, end) -> (travel_time, path)
        self.valid_pairs = []  # List of (start, end) pairs within time limit
        self.precompute_all_paths()

        print("📦 Generating deliveries with pre-computed paths...")
        self.deliveries = self.generate_deliveries()

        print("⏱️  Calculating time limits...")
        self.calculate_time_limits()

        print(f"✅ Generated {len(self.deliveries)} deliveries")

    def generate_distribution(self):
        mu_list = [peak * 60 * self.hours for peak in self.peaks]
        x = np.linspace(0, 60 * self.hours, 60 * self.hours)
        y_combined = np.zeros_like(x)
        for mu in mu_list:
            y_combined += (1 / (self.sigma * np.sqrt(2 * np.pi))) * \
                         np.exp(-0.5 * ((x - mu) / self.sigma) ** 2)
        loads = [int((self.load * (self.sigma * np.sqrt(2 * np.pi))) * y)
                for y in y_combined]
        return y_combined, loads

    def precompute_all_paths(self):
        valid_nodes = list(range(self.num_hotspots))
        total_pairs = len(valid_nodes) * (len(valid_nodes) - 1)

        with tqdm(total=total_pairs, desc="Pre-computing paths") as pbar:
            for start in valid_nodes:
                for end in valid_nodes:
                    if start != end:
                        travel_time, path = astar_shortest_path(
                            adjacency=self.adjacency_matrix,
                            start=start,
                            goal=end,
                            node_positions=self.node_positions,
                            weight_type='time'
                        )
                        self.precomputed_paths[(start, end)] = (travel_time, path)
                        if travel_time <= self.max_delivery_time:
                            self.valid_pairs.append((start, end))
                        pbar.update(1)

        print(f"✓ Pre-computed {len(self.precomputed_paths)} shortest paths")
        print(f"✓ Found {len(self.valid_pairs)} valid delivery pairs (within {self.max_delivery_time}s limit)")

        if not self.valid_pairs:
            raise ValueError(f"No valid delivery pairs found within {self.max_delivery_time}s time limit!")

    def calculate_shortest_path(self, start, end):
        if start == end: return [start]
        if (start, end) not in self.precomputed_paths:
            print(f"⚠️ WARNING: Path ({start}, {end}) not found in pre-computed paths!")
            return [start, end]
        travel_time, path = self.precomputed_paths[(start, end)]
        return path

    def get_travel_time(self, start, end):
        if start == end: return 0
        if (start, end) not in self.precomputed_paths:
            print(f"⚠️ WARNING: Path ({start}, {end}) not found in pre-computed paths!")
            return float('inf')
        travel_time, path = self.precomputed_paths[(start, end)]
        return travel_time

    def calculate_time_limits(self):
        for delivery in self.deliveries:
            travel_time = self.get_travel_time(delivery.start_node, delivery.end_node)
            required_time = travel_time * delivery.TIME_TOLERANCE_FACTOR
            for interval in INTERVALS:
                if required_time <= interval:
                    delivery.time_limit = interval
                    break
            else:
                delivery.time_limit = float('inf')

    def generate_deliveries(self):
        deliveries = []
        total_deliveries = sum(self.loads)
        if not self.valid_pairs:
            raise ValueError("No valid delivery pairs available!")

        with tqdm(total=total_deliveries, desc="Creating deliveries") as pbar:
            for j in range(60 * self.hours):
                for _ in range(self.loads[j]):
                    start, end = random.choice(self.valid_pairs)
                    travel_time, shortest_path = self.precomputed_paths[(start, end)]
                    time = random.randint(j * 60, (j + 1) * 60)
                    delivery = Delivery(
                        start, end, time,
                        time_tolerance_factor=self.time_tolerance_factor,
                        shortest_path=shortest_path
                    )
                    deliveries.append(delivery)
                    pbar.update(1)
        return deliveries

    def reset_deliveries(self):
        """Reset all deliveries to their initial state."""
        Delivery.id_counter = 0 # Reset the global ID counter
        for d in self.deliveries:
            d.reset()
            # Re-assign IDs to be consistent
            d.id = Delivery.id_counter
            Delivery.id_counter += 1

#### 4. Load Graph Data

(This is UNCHANGED from the original file)

In [17]:
print("\n[1] Loading graph and census data...")
try:
    census_df = gpd.read_file(data_dir + "/Census Data/census_tract_data.geojson")
    num_hotspots = len(census_df.index)
    print(f"    ✓ Census tracts loaded: {num_hotspots} tracts (hotspots)")

    # --- CORRECTED FILE PATHS ---
    # Added .xml to the end based on your screenshot
    umst_path = data_dir + "/UMST Graph/graphs/umst_graph.graphml.xml"
    mst_path = data_dir + "/UMST Graph/graphs/mst_graph.graphml.xml"
    hotspot_path = data_dir + "/UMST Graph/graphs/gh_hotspot_graph.graphml.xml"

    umst_graph = nx.read_graphml(umst_path)
    print(f"    ✓ UMST Graph loaded: {umst_graph.number_of_nodes()} nodes, {umst_graph.number_of_edges()} edges")

    # --- 5. Build Adjacency & Position Dictionaries ---
    # (This section is the same as before)
    print("\n[2] Building node mappings and adjacency...")
    nodes = sorted(umst_graph.nodes())
    index_to_tract = {idx: node for idx, node in enumerate(nodes)}
    tract_to_index = {node: idx for idx, node in enumerate(nodes)}

    adjacency_matrix = build_adjacency_matrix(umst_graph, tract_to_index)

    node_positions = {}
    for geo_id, data in umst_graph.nodes(data=True):
        idx = tract_to_index[geo_id]
        lat = float(data.get('lat', 0))
        lon = float(data.get('lon', 0))
        node_positions[idx] = (lat, lon)
    print("    ✓ Node position dictionary built.")

    # --- 6. Instantiate the DeliveryList ---
    # (This section is the same as before)
    print("\n[3] Generating all delivery tasks for the simulation...")
    random.seed(42) # Use a fixed seed for reproducibility
    np.random.seed(42)

    deliverylist = DeliveryList(
        max_delivery_time=1800, # all random deliveries will be under this
        load= 400,
        time_tolerance_factor=2.0,
        hours=1,
        peaks=[0.25, 0.75],
        sigma=10,
        num_hotspots=num_hotspots,
        adjacency_matrix=adjacency_matrix,
        node_positions=node_positions
    )

except Exception as e:
    print(f"\n--- !! ERROR !! ---")
    print(f"Could not load graph data or build DeliveryList.")
    print(f"Make sure your Google Drive is mounted and the path is correct:")
    print(f"PATH: {data_dir}")
    print(f"Error: {e}")


[1] Loading graph and census data...
    ✓ Census tracts loaded: 26 tracts (hotspots)
    ✓ UMST Graph loaded: 26 nodes, 47 edges

[2] Building node mappings and adjacency...
✓ Adjacency built: 26 nodes with 47 edges
    ✓ Node position dictionary built.

[3] Generating all delivery tasks for the simulation...
📊 Generating temporal distribution...
🗺️  Pre-computing all shortest paths...


Pre-computing paths: 100%|██████████| 650/650 [00:00<00:00, 44182.77it/s]


✓ Pre-computed 650 shortest paths
✓ Found 636 valid delivery pairs (within 1800s limit)
📦 Generating deliveries with pre-computed paths...


Creating deliveries: 100%|██████████| 18498/18498 [00:00<00:00, 267278.84it/s]

⏱️  Calculating time limits...
✅ Generated 18498 deliveries


# 3. THE RL ENVIRONMENT & NEW VEHICLE (AGENT) CLASS
 This contains the vehicle objects and the delivery env(It replaces the Simulation class from the `V7_02_order_bundling.ipynb` code.

In [33]:
# ===========================================================
# CHUNK 3 (PATH B, v6): FINAL ENV (Shaping & Bundling Fixed)
# ===========================================================

import heapq # For the failure queue

# --- 1. Agent Class (The Vehicle) ---
# (UNCHANGED from before)
class VehicleAgent:
    def __init__(self, agent_id, start_node, capacity=3):
        self.id = agent_id
        self.start_node = start_node
        self.capacity = capacity
        self.current_node = start_node
        self.status = 0 # 0:IDLE, 1:MOVING_TO_TASK, 2:MOVING_WITH_CARGO, 3:IN_TRANSIT
        self.path = []
        self.path_index = 0
        self.time_till_next_node = 0
        self.cargo = []
        self.distance_traveled = 0
        self.deliveries_completed = 0
        self.prev_dist_to_target = float('inf') # For reward shaping

    def reset(self):
        self.current_node = self.start_node
        self.status = 0
        self.cargo = []
        self.path = []
        self.path_index = 0
        self.time_till_next_node = 0
        self.distance_traveled = 0
        self.deliveries_completed = 0
        self.prev_dist_to_target = float('inf')

    def in_transit(self):
        return self.status == 3

    def is_idle(self):
        return self.status == 0

    def has_capacity(self):
        return len(self.cargo) < self.capacity

    def add_cargo(self, delivery):
        if not self.has_capacity():
            return
        self.cargo.append(delivery)

    def get_cargo_destinations(self):
        return [d.end_node for d in self.cargo]

    def move_to_next_node(self, distance, travel_time):
        self.status = 3 # IN_TRANSIT
        self.time_till_next_node = travel_time
        self.distance_traveled += distance

    def arrive_at_node(self, node):
        self.current_node = node
        self.path_index += 1
        # Determine new status based on path completion
        if not self.path or self.path_index >= len(self.path):
            self.status = 0 # Path complete, now IDLE
            self.path = []
            self.path_index = 0
        elif len(self.cargo) > 0:
            self.status = 2 # Still has cargo, MOVING_WITH_CARGO
        else:
            self.status = 1 # No cargo, MOVING_TO_TASK

    def __repr__(self):
        return f"V{self.id}@N{self.current_node} (Status: {self.status}, Cargo: {len(self.cargo)}/{self.capacity})"


# --- 2. The New *FINAL* Fleet Environment ---
class FleetEnv:
    TIME_STEP_SIZE = 10
    N_AGENTS = 100
    VEHICLE_CAPACITY = 3

    REWARD_DELIVERED = 100.0
    REWARD_PICKUP = 10.0
    REWARD_FAILED = -100.0
    REWARD_TIME_STEP = -0.01
    REWARD_MOVE_STEP = -0.01
    REWARD_IDLE_PENALTY = -0.1
    K_POTENTIAL = 2.0 # Reward shaping scalar
    GAMMA = 0.99 # Must match agent's gamma

    STATE_SIZE = 26 + 1 + VEHICLE_CAPACITY + 1 + 1 + 26 + 1 # Total = 59
    ACTION_SIZE = 2 # 0: STAY_IDLE, 1: GO_TO_TARGET

    def __init__(self, deliverylist, adjacency_matrix, node_positions, num_hotspots):
        print(f"\n[4] Initializing *FINAL Fleet-based* Environment (v6)...")
        self.deliverylist = deliverylist
        self.adjacency_matrix = adjacency_matrix
        self.node_positions = node_positions
        self.num_hotspots = num_hotspots

        self.simulation_end_time = self.deliverylist.hours * 3600 + 900

        self.all_tasks = sorted(self.deliverylist.deliveries, key=lambda d: d.start_time)
        self.task_pointer = 0
        self.available_tasks = {}
        self.node_to_tasks = defaultdict(set)
        self.task_failure_queue = []

        self.vehicle_agents = {i: VehicleAgent(i, random.randint(0, self.num_hotspots - 1), self.VEHICLE_CAPACITY) for i in range(self.N_AGENTS)}
        self.node_to_agents = defaultdict(set)
        # Note: reset() will handle initial placement

        self.clock = 0
        self.total_vehicle_distance = 0
        self.completed_deliveries = 0
        self.failed_deliveries = 0
        print(f"    ✓ Environment created with {self.N_AGENTS} fleet agents.")
        print(f"    ✓ Environment created with {len(self.all_tasks)} total tasks.")
        print(f"    ✓ State Size: {self.STATE_SIZE}, Action Size: {self.ACTION_SIZE}")

    def _get_edge_info(self, from_node, to_node):
        if from_node == to_node: return 0, 0
        if from_node not in self.adjacency_matrix: return None, None
        for neighbor_idx, dist, time in self.adjacency_matrix[from_node]:
            if neighbor_idx == to_node:
                return dist, time
        return None, None

    def _get_path(self, start_node, end_node):
        if start_node == end_node: return [start_node]
        return self.deliverylist.calculate_shortest_path(start_node, end_node)

    # *** NEW/FIXED HELPER FUNCTIONS ***
    def _find_nearest_task_node(self, agent_node):
        """Finds the node ID of the nearest available task."""
        if not self.available_tasks: return -1
        min_dist = float('inf')
        nearest_node = -1
        for node_id in self.node_to_tasks:
            if not self.node_to_tasks[node_id]: continue
            time_to_node = self.deliverylist.get_travel_time(agent_node, node_id)
            if time_to_node < min_dist:
                min_dist = time_to_node
                nearest_node = node_id
        return nearest_node

    def _find_nearest_task_dist(self, agent_node):
        """Helper for reward shaping: finds distance to nearest task."""
        nearest_node = self._find_nearest_task_node(agent_node)
        if nearest_node == -1:
            return 0 # No tasks, no potential
        dist = self.deliverylist.get_travel_time(agent_node, nearest_node)
        return dist if dist != float('inf') else 0

    def _get_agent_state(self, agent_id):
        """Builds the 59-dimensional state vector."""
        agent = self.vehicle_agents[agent_id]

        node_vec = self._to_one_hot(agent.current_node, self.num_hotspots)

        cargo_status = len(agent.cargo) / agent.capacity

        cargo_dests = np.zeros(self.VEHICLE_CAPACITY)
        for i, task in enumerate(agent.cargo):
            cargo_dests[i] = task.end_node / self.num_hotspots

        time_vec = [self.clock / self.simulation_end_time]

        tasks_at_node = [min(len(self.node_to_tasks.get(agent.current_node, set())) / 5.0, 1.0)]

        # --- 6. Nearest task location (one-hot, 26 dims) & update potential ---
        # *** THIS IS THE FIX ***
        nearest_task_node_id = self._find_nearest_task_node(agent.current_node)
        nearest_task_vec = self._to_one_hot(nearest_task_node_id, self.num_hotspots)

        # For reward shaping, we calculate and store the "potential"
        if agent.is_idle() and not agent.has_capacity():
            # If idle and full, potential is dist to *dropoff*
            dists = [self.deliverylist.get_travel_time(agent.current_node, d.end_node) for d in agent.cargo]
            agent.current_dist_to_target = min(dists) if dists else 0
        else:
            # If idle and not full, potential is dist to *pickup*
            agent.current_dist_to_target = self._find_nearest_task_dist(agent.current_node)

        other_agents = (len(self.node_to_agents.get(agent.current_node, set())) - 1)
        other_agents_vec = [min(other_agents / (self.N_AGENTS - 1 + 1e-6), 1.0)]

        state = np.concatenate([
            node_vec, [cargo_status], cargo_dests, time_vec,
            tasks_at_node, nearest_task_vec, other_agents_vec
        ])

        return np.array(state)

    def _to_one_hot(self, index, size):
        vec = np.zeros(size)
        if 0 <= index < size:
            vec[index] = 1.0
        return vec

    def _activate_new_tasks(self):
        for i in range(self.task_pointer, len(self.all_tasks)):
            task = self.all_tasks[i]
            if task.start_time <= self.clock:
                self.available_tasks[task.id] = task
                self.node_to_tasks[task.start_node].add(task.id)
                heapq.heappush(self.task_failure_queue, (task.time_limit, task.id))
                self.task_pointer = i + 1
            else:
                break

    def _check_failed_tasks(self):
        while self.task_failure_queue and self.task_failure_queue[0][0] <= self.clock:
            time_limit, task_id = heapq.heappop(self.task_failure_queue)
            if task_id in self.available_tasks:
                task = self.available_tasks.pop(task_id)
                self.node_to_tasks[task.start_node].discard(task_id)
                task.completed = True
                task.successful = False
                self.failed_deliveries += 1

    def reset(self):
        self.deliverylist.reset_deliveries()
        self.all_tasks = sorted(self.deliverylist.deliveries, key=lambda d: d.start_time)
        self.task_pointer = 0
        self.available_tasks = {}
        self.node_to_tasks = defaultdict(set)
        self.task_failure_queue = []
        self.node_to_agents = defaultdict(set)
        for i, agent in self.vehicle_agents.items():
            start_node = random.randint(0, self.num_hotspots - 1)
            agent.start_node = start_node
            agent.reset()
            # --- THIS IS THE FIX ---
            agent.prev_dist_to_target = self._find_nearest_task_dist(agent.current_node)
            self.node_to_agents[agent.current_node].add(agent.id)
        self.clock = 0
        self.total_vehicle_distance = 0
        self.completed_deliveries = 0
        self.failed_deliveries = 0
        self._activate_new_tasks()
        initial_states = {
            agent_id: self._get_agent_state(agent_id)
            for agent_id in self.vehicle_agents.keys()
        }
        return initial_states

    def step(self, actions):
        self.clock += self.TIME_STEP_SIZE
        rewards = defaultdict(float)
        dones = defaultdict(lambda: False)

        self._activate_new_tasks()
        self._check_failed_tasks()

        agents_arriving = []
        for agent_id in self.vehicle_agents.keys():
            agent = self.vehicle_agents[agent_id]
            rewards[agent_id] += self.REWARD_TIME_STEP
            if agent.in_transit():
                rewards[agent_id] += self.REWARD_MOVE_STEP
                agent.time_till_next_node -= self.TIME_STEP_SIZE
                if agent.time_till_next_node <= 0:
                    agents_arriving.append(agent)

        for agent in agents_arriving:
            if agent.id not in self.vehicle_agents: continue

            self.node_to_agents[agent.current_node].discard(agent.id)

            if not agent.path or agent.path_index >= len(agent.path):
                agent.arrive_at_node(agent.current_node) # Calls arrive_at_node to set status=0
            else:
                next_node_in_path = agent.path[agent.path_index]
                agent.arrive_at_node(next_node_in_path)

            self.node_to_agents[agent.current_node].add(agent.id)

            delivered_cargo = []
            for task in agent.cargo:
                if agent.current_node == task.end_node:
                    task.completed = True
                    task.end_time = self.clock
                    if task.end_time <= task.time_limit:
                        task.successful = True
                        rewards[agent.id] += self.REWARD_DELIVERED
                    else:
                        task.successful = False
                        rewards[agent.id] += self.REWARD_FAILED
                    self.completed_deliveries += 1
                    agent.deliveries_completed += 1
                    delivered_cargo.append(task)
            agent.cargo = [task for task in agent.cargo if task not in delivered_cargo]

            if agent.path_index >= len(agent.path):
                 agent.status = 0
                 agent.path = []
                 agent.path_index = 0

        for agent_id, action in actions.items():
            agent = self.vehicle_agents[agent_id]
            if agent.in_transit() or not agent.is_idle():
                continue

            # --- Get distance for reward shaping ---
            d_prev = agent.prev_dist_to_target

            self.node_to_agents[agent.current_node].discard(agent.id)

            # --- Auto-pickup logic ---
            tasks_at_node = self.node_to_tasks.get(agent.current_node, set())
            while agent.has_capacity() and tasks_at_node:
                task_id = tasks_at_node.pop()
                if task_id in self.available_tasks:
                    task = self.available_tasks.pop(task_id)
                    agent.add_cargo(task)
                    rewards[agent.id] += self.REWARD_PICKUP

            # --- Action 0: STAY_IDLE ---
            if action == 0:
                if not self.node_to_tasks[agent.current_node] and not agent.cargo:
                    rewards[agent.id] += self.REWARD_IDLE_PENALTY
                self.node_to_agents[agent.current_node].add(agent.id)
                agent.prev_dist_to_target = d_prev
                continue

            # --- Action 1: GO_TO_TARGET ---
            target_node = -1
            if agent.has_capacity():
                target_node = self._find_nearest_task_node(agent.current_node)
            else: # If full, go to nearest dropoff
                dists = [(self.deliverylist.get_travel_time(agent.current_node, d.end_node), d.end_node) for d in agent.cargo]
                if dists:
                    target_node = min(dists)[1]

            if target_node == -1 or target_node == agent.current_node:
                rewards[agent.id] += self.REWARD_IDLE_PENALTY
                self.node_to_agents[agent.current_node].add(agent.id)
                agent.prev_dist_to_target = d_prev
                continue

            agent.path = self._get_path(agent.current_node, target_node)
            agent.path_index = 0
            agent.status = 1 if agent.has_capacity() else 2

            if not agent.path or len(agent.path) < 2:
                agent.status = 0; self.node_to_agents[agent.current_node].add(agent.id); continue

            next_node_in_path = agent.path[agent.path_index]
            if agent.current_node == next_node_in_path:
                agent.path_index += 1
                if agent.path_index >= len(agent.path):
                    agent.status = 0; self.node_to_agents[agent.current_node].add(agent.id); continue
                next_node_in_path = agent.path[agent.path_index]

            dist, time = self._get_edge_info(agent.current_node, next_node_in_path)
            if dist is not None and time > 0:
                agent.move_to_next_node(dist, time)
                self.total_vehicle_distance += dist

                # --- Reward Shaping (Friend's A.1) ---
                d_next = self.deliverylist.get_travel_time(next_node_in_path, target_node)
                phi_prev = -d_prev
                phi_next = -d_next
                shaping = self.K_POTENTIAL * ( (self.GAMMA * phi_next) - phi_prev )
                rewards[agent.id] += shaping
                agent.prev_dist_to_target = d_next
            else:
                agent.status = 0
                self.node_to_agents[agent.current_node].add(agent.id)

        next_states = {
            agent_id: self._get_agent_state(agent_id)
            for agent_id in self.vehicle_agents.keys()
        }

        is_done_all = False
        if (self.task_pointer == len(self.all_tasks) and not self.available_tasks and all(a.is_idle() for a in self.vehicle_agents.values())) or \
           (self.clock >= self.simulation_end_time):
            is_done_all = True
            for agent_id in self.vehicle_agents.keys():
                dones[agent_id] = True

        dones['__all__'] = is_done_all
        info = {
            'active_tasks': len(self.available_tasks),
            'completed': self.completed_deliveries,
            'failed': self.failed_deliveries
        }
        return next_states, rewards, dones, info


# 4.

In [34]:
# ===========================================================
# CHUNK 4 (PATH B, v5): "MADDPG" AGENT (DQN-BASED)
# ===========================================================

# --- 1. Hyperparameters for DQN ---
BUFFER_SIZE = int(1e5)  # Replay buffer size
BATCH_SIZE = 128        # Minibatch size
GAMMA = 0.99            # Discount factor (Must match env's GAMMA)
TAU = 1e-3              # For soft update of target parameters
LR = 5e-4               # Learning rate
UPDATE_EVERY = 4        # How often to update the network
LEARN_TIMES = 5         # Number of times to learn per update

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# --- 2. The Q-Network (The "Brain") ---
class QNetwork(nn.Module):
    """Action-Value Model."""
    def __init__(self, state_size, action_size, seed, fc1_units=128, fc2_units=64):
        super(QNetwork, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, action_size)

    def forward(self, state):
        """Build a network that maps state -> action values."""
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# --- 3. Replay Buffer (The "Memory") ---
class ReplayBuffer:
    def __init__(self, action_size, buffer_size, batch_size, seed):
        self.action_size = action_size
        self.memory = deque(maxlen=buffer_size)
        self.batch_size = batch_size
        self.experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])
        self.seed = random.seed(seed)

    def add(self, state, action, reward, next_state, done):
        e = self.experience(state, action, reward, next_state, done)
        self.memory.append(e)

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)
        states = torch.from_numpy(np.vstack([e.state for e in experiences if e is not None])).float().to(device)
        actions = torch.from_numpy(np.vstack([e.action for e in experiences if e is not None])).long().to(device)
        rewards = torch.from_numpy(np.vstack([e.reward for e in experiences if e is not None])).float().to(device)
        next_states = torch.from_numpy(np.vstack([e.next_state for e in experiences if e is not None])).float().to(device)
        dones = torch.from_numpy(np.vstack([e.done for e in experiences if e is not None]).astype(np.uint8)).float().to(device)
        return (states, actions, rewards, next_states, dones)

    def __len__(self):
        return len(self.memory)

# --- 4. The Agent Class (The "Decision Maker") ---
class DQNAgent():
    """Interacts with and learns from the environment."""

    def __init__(self, state_size, action_size, seed):
        """Initialize an Agent object."""
        self.state_size = state_size
        self.action_size = action_size
        self.seed = random.seed(seed)
        self.qnetwork_local = QNetwork(state_size, action_size, seed).to(device)
        self.qnetwork_target = QNetwork(state_size, action_size, seed).to(device)
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=LR)
        self.memory = ReplayBuffer(action_size, BUFFER_SIZE, BATCH_SIZE, seed)
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        self.memory.add(state, action, reward, next_state, done)
        self.t_step = (self.t_step + 1) % UPDATE_EVERY
        if self.t_step == 0:
            if len(self.memory) > BATCH_SIZE:
                for _ in range(LEARN_TIMES):
                    experiences = self.memory.sample()
                    self.learn(experiences, GAMMA)

    def act(self, state, eps=0.):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        self.qnetwork_local.eval()
        with torch.no_grad():
            action_values = self.qnetwork_local(state)
        self.qnetwork_local.train()

        if random.random() > eps:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self, experiences, gamma):
        states, actions, rewards, next_states, dones = experiences
        Q_targets_next = self.qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
        Q_targets = rewards + (gamma * Q_targets_next * (1 - dones))
        Q_expected = self.qnetwork_local(states).gather(1, actions)
        loss = F.mse_loss(Q_expected, Q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.soft_update(self.qnetwork_local, self.qnetwork_target, TAU)

    def soft_update(self, local_model, target_model, tau):
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(tau*local_param.data + (1.0-tau)*target_param.data)

    def save_models(self, path):
        """Saves the shared Q-network."""
        torch.save(self.qnetwork_local.state_dict(), os.path.join(path, 'qnetwork.pth'))
        print(f"\nShared Q-Network saved to {path}")

    def load_models(self, path):
        """Loads the shared Q-network."""
        self.qnetwork_local.load_state_dict(torch.load(os.path.join(path, 'qnetwork.pth')))
        self.qnetwork_target.load_state_dict(torch.load(os.path.join(path, 'qnetwork.pth')))
        print(f"Shared Q-Network loaded from {path}")


# --- 5. Instantiate the Environment and *Shared* Agent ---
try:
    # We must have the 'FleetEnv' class from Chunk 3 in memory
    if 'FleetEnv' not in globals():
        raise NameError("'FleetEnv' not defined. Please re-run Chunk 3.")
    if 'deliverylist' not in globals():
        raise NameError("'deliverylist' not defined. Please re-run Chunk 2.")

    # Instantiate the *evaluation* environment (full 18k tasks)
    eval_env = FleetEnv(deliverylist, adjacency_matrix, node_positions, num_hotspots)

    # Instantiate the *SINGLE* agent that all 100 vehicles will share
    agent = DQNAgent(
        state_size=eval_env.STATE_SIZE,
        action_size=eval_env.ACTION_SIZE,
        seed=42
    )

    print("    ✓ DQN (Shared, 2-Action) Agent and Environment instantiated successfully.")

except Exception as e:
    print(f"\n--- !! ERROR !! ---")
    print(f"Could not instantiate environment or agent.")
    print(f"Error: {e}")
    traceback.print_exc()

Using device: cuda

[4] Initializing *FINAL Fleet-based* Environment (v6)...
    ✓ Environment created with 100 fleet agents.
    ✓ Environment created with 18498 total tasks.
    ✓ State Size: 59, Action Size: 2
    ✓ DQN (Shared, 2-Action) Agent and Environment instantiated successfully.


# 5. THE MADDPG TRAINING LOOP

In [ ]:
# ===========================================================
# CHUNK 5 (PATH B, v5): THE *FINAL* TRAINING LOOP
# ===========================================================
import traceback # Import the traceback library
from collections import deque, defaultdict
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import copy
from collections import namedtuple, deque

# --- 1. Create a smaller "Training" Environment ---
# We must re-run this to ensure the `train_env` and `agent` objects
# are the ones we intend to use for training.

print("\n[5] Generating smaller 'Training' delivery list...")
random.seed(123) # Use a different seed for the training set
np.random.seed(123)

try:
    # We must have deliverylist defined from Chunk 2
    if 'deliverylist' not in globals():
        raise NameError("'deliverylist' not defined. Please re-run Chunk 2.")
    if 'FleetEnv' not in globals():
        raise NameError("'FleetEnv' not defined. Please re-run Chunk 3.")
    if 'DQNAgent' not in globals():
        raise NameError("'DQNAgent' not defined. Please re-run Chunk 4.")


    deliverylist_train = DeliveryList(
        max_delivery_time=1800,
        load=100,  # <-- Smaller load for faster training
        time_tolerance_factor=2.0,
        hours=1,
        peaks=[0.25, 0.75],
        sigma=10,
        num_hotspots=num_hotspots,
        adjacency_matrix=adjacency_matrix,
        node_positions=node_positions
    )

    # Instantiate the training environment
    train_env = FleetEnv(deliverylist_train, adjacency_matrix, node_positions, num_hotspots)

    print(f"\n    ✓ Training environment created with {train_env.N_AGENTS} agents and {len(train_env.all_tasks)} tasks.")

    # Re-initialize the agent
    agent = DQNAgent(
        state_size=train_env.STATE_SIZE,
        action_size=train_env.ACTION_SIZE,
        seed=42
    )
    print("    ✓ Shared DQN Agent re-initialized for training.")

except Exception as e:
    print(f"\n--- !! ERROR during setup !! ---")
    print(f"Error: {e}")
    traceback.print_exc()


# --- 2. The Training Function ---
def dqn_train(n_episodes=750, print_every=10, eps_start=1.0, eps_end=0.01, eps_decay=0.995):
    """Main loop to train the DQN agent."""

    scores_deque = deque(maxlen=print_every)
    scores_history = []
    eps = eps_start

    print("\n[6] Starting DQN Training...")

    for i_episode in range(1, n_episodes + 1):
        states = train_env.reset() # This is now a dict {agent_id: state}
        episode_rewards = defaultdict(float) # Tracks score per agent

        # We loop for a max number of timesteps to prevent infinite loops
        max_t = (train_env.simulation_end_time // train_env.TIME_STEP_SIZE) + 1

        for t in range(max_t):

            # --- 1. Agents Act ---
            # Get actions for all 100 agents
            actions_to_take = {}
            for agent_id in train_env.vehicle_agents.keys():
                # Only idle agents get to make a "dispatch" decision
                if train_env.vehicle_agents[agent_id].is_idle():
                    state = states[agent_id]
                    actions_to_take[agent_id] = agent.act(state, eps)

            # --- 2. Environment Steps ---
            next_states, rewards, dones, info = train_env.step(actions_to_take)

            # --- 3. Store Experiences & Learn ---
            # *** THIS IS THE FIX ***
            # We ONLY iterate over the agents that actually took an action
            for agent_id in actions_to_take.keys():
                state = states[agent_id]
                action = actions_to_take[agent_id]
                reward = rewards.get(agent_id, 0.0) # Get this agent's reward
                next_state = next_states[agent_id] # Get this agent's next state
                done = dones.get(agent_id, False) # Get this agent's done flag

                episode_rewards[agent_id] += reward

                # Add to shared buffer.
                # agent.step() will handle the UPDATE_EVERY and learn() logic.
                agent.step(state, action, reward, next_state, done)

            # We also must store experiences for agents that were *not* idle
            # so they learn the cost of moving/waiting
            for agent_id in train_env.vehicle_agents.keys():
                if agent_id not in actions_to_take:
                    state = states[agent_id]
                    action = 0 # Placeholder for "no decision"
                    reward = rewards.get(agent_id, 0.0)
                    next_state = next_states[agent_id]
                    done = dones.get(agent_id, False)

                    episode_rewards[agent_id] += reward
                    # We still call step() to add to memory and trigger learning
                    agent.step(state, action, reward, next_state, done)


            states = next_states

            # --- 4. Check for End of Episode ---
            if dones.get('__all__', False):
                break

        # --- End of Episode ---
        eps = max(eps_end, eps_decay * eps) # Decay epsilon

        # Calculate total score from ALL agents
        total_episode_score = sum(episode_rewards.values())

        scores_deque.append(total_episode_score)
        scores_history.append(total_episode_score)
        avg_score = np.mean(scores_deque)

        print(f'\rEpisode {i_episode}\tAverage Score: {avg_score:.2f}\tTotal Score: {total_episode_score:.2f}\tEpsilon: {eps:.3f}', end="")

        if i_episode % print_every == 0:
            print(f'\rEpisode {i_episode}\tAverage Score: {avg_score:.2f}\tCompleted: {info["completed"]}\tFailed: {info["failed"]}')
            # Save a checkpoint
            agent.save_models(output_dir)

    # --- End of Training ---
    print(f'\nTraining complete. Final Average Score: {np.mean(scores_deque):.2f}')
    agent.save_models(output_dir) # Save final models
    return scores_history


# --- 3. Run the Training ---
try:
    # Run for 750 episodes, print every 10
    scores = dqn_train(n_episodes=750, print_every=10)

    # --- 4. Plot Training Scores ---
    print("\n[7] Plotting Training Progress...")
    fig = plt.figure()
    ax = fig.add_subplot(111)
    plt.plot(np.arange(1, len(scores)+1), scores)
    plt.ylabel('Total Episode Score')
    plt.xlabel('Episode #')
    plt.title('Total Reward per Training Episode (Fleet Model)')
    plt.savefig(os.path.join(output_dir, 'training_scores_fleet.png'))
    plt.show()

except Exception as e:
    print(f"\n--- !! ERROR during Training !! ---")
    print(f"Error: {e}")
    traceback.print_exc()
    print("Training was interrupted.")


[5] Generating smaller 'Training' delivery list...
📊 Generating temporal distribution...
🗺️  Pre-computing all shortest paths...


Pre-computing paths: 100%|██████████| 650/650 [00:00<00:00, 43064.03it/s]


✓ Pre-computed 650 shortest paths
✓ Found 636 valid delivery pairs (within 1800s limit)
📦 Generating deliveries with pre-computed paths...


Creating deliveries: 100%|██████████| 4604/4604 [00:00<00:00, 251452.88it/s]

⏱️  Calculating time limits...
✅ Generated 4604 deliveries

[4] Initializing *FINAL Fleet-based* Environment (v6)...
    ✓ Environment created with 100 fleet agents.
    ✓ Environment created with 4604 total tasks.
    ✓ State Size: 59, Action Size: 2

    ✓ Training environment created with 100 agents and 4604 tasks.
    ✓ Shared DQN Agent re-initialized for training.

[6] Starting DQN Training...


# 6.